In [ ]:
import torch 
import pickle
from pathlib import Path
import yaml
from model import GPT, GPTConfig


# load reward model for evaluating nanoGPT responses
base_path = Path.cwd()

meta_data = base_path/ "meta.pkl"
meta_data = pickle.load(open(meta_data, "rb"))
with open('config/config_reward.yaml') as f:
    conf = yaml.load(f, Loader=yaml.FullLoader)
    # nested dictionary structure
    config = {}               
    for k, v in conf.items():
        for k2, v2 in v.items():
            config[k2] = v2

vocab_size = meta_data['vocab_size']
model_args = dict(n_layer=config['n_layer'], n_head=config['n_head'], n_embd=config['n_embd'], block_size=config['block_size'],
                    bias=config['bias'], vocab_size=vocab_size, dropout=config['dropout'], ) # start with model_args from command line
reward_model = GPT(GPTConfig(**model_args))
reward_model_dict = torch.load(base_path/"reward_model.pth", map_location="cpu")
reward_model.load_state_dict(reward_model_dict)
reward_model = reward_model.cuda()
reward_model.eval()

#decoder
def evaluate_response(prompt):
    encoded = [ meta_data['stoi'][ch]  for ch in prompt]
    if len(encoded) == 0:
        return 0.0
    inputs = torch.tensor(encoded, dtype=torch.long).unsqueeze(0)  # Add batch dimension
    device = next(reward_model.parameters()).device
    inputs = inputs.to(device)
    with torch.no_grad():
        outputs = reward_model(inputs)
    reward_score = outputs.item()
    return reward_score

{'out_dir': 'out', 'eval_interval': 10, 'log_interval': 1, 'eval_iters': 20, 'eval_only': False, 'always_save_checkpoint': True, 'init_from': 'resume', 'init_multihead_from': 'scratch', 'out_dir_multihead': 'out_reward', 'wandb_log': True, 'wandb_project': 'rlhf', 'wandb_run_name': 'gpt2', 'dataset': 'shakespeare', 'gradient_accumulation_steps': 1, 'batch_size': 64, 'block_size': 256, 'n_layer': 2, 'n_head': 2, 'n_embd': 768, 'dropout': 0.0, 'bias': False, 'learning_rate': 0.0006, 'max_iters': 20, 'weight_decay': 0.01, 'beta1': 0.9, 'beta2': 0.95, 'grad_clip': 1.0, 'decay_lr': True, 'warmup_iters': 2000, 'lr_decay_iters': 600000, 'min_lr': 6e-05, 'backend': 'nccl', 'device': 'cuda', 'dtype': 'float16', 'compile': False}
number of parameters: 14.29M


/tmp/ipykernel_1262036/2343731216.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  reward_model_dict = torch.load(base_path/"reward_model.pth", map_location="cpu")


In [ ]:
shakspeare_sample  =  """
Men yet he shall show me that hath not held to see.

ROMEO:
Farewell; sweet love, and get thee hence; for I must not;
For I am sorry, come not hither at my last.

FRIAR LAURENCE:
My lord, so hate the ground is ashame.

FRIAR LAURENCE:
So long as I love the dangerous tongue.

ROMEO:
O Rosaline, make me not: I love the duke
From Rosaline, till I prove a honour of your
Love of his own design. O Romeo, Romeo!

ROMEO:
What say you?

FRIAR JOHN:
No, by and by?

FRIAR LAURENCE:
Blessed of foot, how our
"""


sample = shakspeare_sample.split('\n')

In [39]:
reward_sample_scores = [ evaluate_response(s) for s in sample]

In [40]:
for sentence,score in zip(sample, reward_sample_scores):
    print(f"{sentence} = {score}")

 = 0.0
Clown: = 1.0501689910888672
So you will be long to be her bed. = 1.4994946718215942
 = 0.0
AUTOLYCUS: = 1.0245933532714844
Softly, awhile shall be the best! I do repent me = 2.0770912170410156
here and to her friends; who, to the gods know I = 1.9578185081481934
should gracely bear my part, that I can do think. = 2.213855028152466
 = 0.0
First Citizen: = 0.7155227661132812
The gods keep you good word to prison! The nobles could = 2.697669267654419
I'll tell you what you are, and what you have done = 2.346397876739502
Come hither against the tackle's voice: but, I = 2.517284631729126
pray you, in whom dance, I would prove a name, if = 2.380936861038208
heaven speak against my crown, I must not think. = 2.1096391677856445
 = 0.0
SICINIUS: = 1.0046942234039307
Go about it. = 1.1291663646697998
 = 0.0
BRU = 0.6744412183761597
 = 0.0
